# Stacking

This notebook implements a stacking ensemble learning technique (meta-learning) for fake news detection.

# Optimization Problem

MIN Z

$$
Z = - \frac{1}{N} \displaystyle\sum_{i=1}^{N} α_{c_i} [w_1 y_i log(F(x_i)) + w_0 (1 - y_i) log(1 - F(x_i))] + λ Ω(θ)
$$

<br>SUBJECT TO <br><br>

$C_1$: Stacking meta-learner

$$F(x) = g(f_1(x), f_2(x), ⋯ , f_M(x))$$

$C_2$: Feature Mapping

$$x_i = ϕ (title_i, text_i, category_i, dataset_i)$$

$C_3$: Binary Labels

$$y_i ∈ \{0, 1\}$$

$C_4$: Category Weight

$$α_{c} = \frac{1}{freq(c_i)}$$

$C_5$: Class Weight

$$w_1 = \frac{N}{2N_1}, \qquad w_0 = \frac{N}{2N_0}$$


# Environment Configuration

The following code cell contains the dependencies that will be used in this notebook.

In [ ]:
# Standard library imports
import warnings
import copy
import cudf
import scipy.sparse as sp
import pandas as pd
import numpy as np

# Third party imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, classification_report, confusion_matrix
)

from sklearn.linear_model import SGDClassifier as skSGD

from cuml.linear_model import LogisticRegression as cuLR
from cuml.naive_bayes import ComplementNB as cuCNB
from cuml.neighbors import KNeighborsClassifier as cuKNN
from cuml.ensemble import RandomForestClassifier as cuRF



NOTE: Some CPU-only components like `TfidfVectorizer`, `OneHotEncoder` and `CalibratedClassifierCV` have no cuML equivalents with full sparse-matrix + `sample_weight` support so they remain on the CPU.

In [2]:
# Utility: Safely convert cuDF Series or CuPy array to NumPy
def to_numpy(arr):
    if hasattr(arr, 'to_numpy'):
        return arr.to_numpy() # cuDF Series / cupy-backed cuDF
    if hasattr(arr, 'get'):
        return arr.get() # cupy.ndarray
    return np.asarray(arr)

# Colab Configuration

Run the code cell below to download data ingestion script from Github.

In [3]:
!wget https://raw.githubusercontent.com/3608Team10/COMP3608PROJECT/refs/heads/main/ingest_data.py

--2026-04-21 21:52:32--  https://raw.githubusercontent.com/3608Team10/COMP3608PROJECT/refs/heads/main/ingest_data.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10317 (10K) [text/plain]
Saving to: ‘ingest_data.py.1’

ingest_data.py.1    100%[===================>]  10.08K  --.-KB/s    in 0s      

2026-04-21 21:52:33 (78.0 MB/s) - ‘ingest_data.py.1’ saved [10317/10317]



# Data Ingestion

Load all fake news datasets using the shared 'ingest_data.py' script. <br>

The resulting dataframe `df` has five columns:
- `title` : title of news
- `text` : actual text content of news
- `category` : type of news
- `label` : 0 = fake, 1 = real
- `dataset` : source dataset of the news record

Duplicate and null text rows are removed during ingestion resulting in ~99K records across three datasets and eight original columns.

In [4]:
from ingest_data import load_datasets

pdf = load_datasets()
df = cudf.from_pandas(pdf)
del pdf

df.head()

/home/johnny/miniforge3/envs/rapids-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


------------------------------------------------------------
Fake News Dataset Ingestion
------------------------------------------------------------

Loading bhavikjikadara ...
[bhavik] Loaded 'true.csv': 21417 rows
[bhavik] Loaded 'fake.csv': 23481 rows

Loading mahdimashayekhi ...
[mahdi] Loaded 'fake_news_dataset.csv': 20000 rows

Loading shawkyelgendy ...
[shawky] Loaded 'real.csv': 21869 rows
[shawky] Loaded 'fake.csv': 19999 rows

Dropped 648 rows with empty/null text.
Dropped 6,650 duplicate rows.

------------------------------------------------------------
Fake News Dataset Summary
------------------------------------------------------------
Total rows: 99,468
Fake (0): 47,089
Real (1): 52,379

Rows per source:
shawkyelgendy          40,824
bhavikjikadara         38,644
mahdimashayekhi        20,000

Categories:
  Sports                 43,691
  Politics               21,635
  News                   19,811
  Health                 2,922
  Entertainment          2,889
  Techno

,title,text,label,category,dataset
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,1,Politics,bhavikjikadara
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,1,Politics,bhavikjikadara
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,1,Politics,bhavikjikadara
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,1,Politics,bhavikjikadara
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,1,Politics,bhavikjikadara


# Text Cleaning and Preprocessing Pipeline

Raw news text contains noise that degrades quality of TF-IDF. <br>

We concatenate `title` and `text` into a single `content` field, then apply a cleaning pipeline that:
- Converts all text to lower case
- Removes URLs
- Strips HTML entities and HTML tags
- Removes punctuation and digits
- Collapses whitespace

Note: Stemming/Lemmatisation is NOT applied at this stage. TF-IDF witha sublinear term-frequency scale handles term frequency inflation naturally <br>

The cleaned text is stored in the new column `content`.

In [5]:
def clean_text(series: cudf.Series) -> cudf.Series:
    series = series.str.lower()
    series = series.str.replace(r'https?://\S+|www\.\S+', ' ', regex=True) # Remove URLs
    series = series.str.replace(r'<[^>]+>', ' ', regex=True) # Strip HTML Tags
    series = series.str.replace(r'&[a-z]+;', ' ', regex=True) # Strip HTML Entities
    series = series.str.replace(r'[^a-z\s]', ' ', regex=True) # Keep letters only
    series = series.str.replace(r'\s+', ' ', regex=True) # Collapse whitespace
    series = series.str.strip()
    return series

df['content'] = clean_text(
    df['title'].fillna('') + ' ' + df['text'].fillna('')
)
df = df.reset_index(drop=True)

df.head()


,title,text,label,category,dataset,content
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,1,Politics,bhavikjikadara,as u s budget fight looms republicans flip the...
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,1,Politics,bhavikjikadara,u s military to accept transgender recruits on...
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,1,Politics,bhavikjikadara,senior u s republican senator let mr mueller d...
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,1,Politics,bhavikjikadara,fbi russia probe helped by australian diplomat...
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,1,Politics,bhavikjikadara,trump wants postal service to charge much more...


# TF-IDF Feature Extraction

$C_2$: $x_i = ϕ(title_i, text_i, category_i, dataset_i)$

Constraint $C_2$ maps raw inputs to a feature vector. The primary signal comes from TF-IDF on `content`. Use:

<pre>
sublinear_tf = True               - Dampens high-frequency terms
max_features = 100 000            - Vocabulary cap to control dimensionality
ngram_range = (1,2)               - unigrams + bigrams to capture short phrases
min_df = 3                        - ignore very rare terms
strip_accents / analyzer = 'word' - standard tokenization
</pre>

The vectorizer is fitted only on the training split to prevent data leakage. At this stage fit using the full dataframe to obtain the index. The fit is to be redone after train/test splitting.

In [6]:
TFIDF_PARAMS = {
    'sublinear_tf': True,
    'max_features': 100000,
    'ngram_range': (1, 2),
    'min_df': 3,
    'strip_accents': 'unicode',
    'analyzer': 'word',
    'token_pattern': r'\b[a-z]{2,}\b' # min 2 characters per word
}

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

print("TF-IDF Vectorizer Configured")
print(f"    ngram_range  : {tfidf.ngram_range}")
print(f"    max_features : {tfidf.max_features:,}")
print(f"    sublinear_tf : {tfidf.sublinear_tf}")

TF-IDF Vectorizer Configured
    ngram_range  : (1, 2)
    max_features : 100,000
    sublinear_tf : True


# Categorical Encoding

The `category` and `dataset` columns are categorical. <br>
One-hot encode these columns and append them as sparse columns to the TF-IDF matrix so the full feature vector satisfies $C_2$. <br>
One-hot encoding is chosen over ordinal encoding because there is no inherent ordering among categories. <br>
We use `sparse_output=True` to keep memory usage manageable when concatenating with the TF-IDF matrix.

In [7]:
cat_encoder = OneHotEncoder(sparse_output=True, handle_unknown='ignore')

cat_encoder.fit(df[['category', 'dataset']].to_pandas())
cat_features_all = cat_encoder.transform(df[['category', 'dataset']].to_pandas())

print("Categorical Encoder Fitted")
print(f"    Input columns   : category, dataset")
print(f"    Output shape    : {cat_features_all.shape}")
print(f"    Category values : {cat_encoder.categories_[0].tolist()}")
print(f"    Dataset values  : {cat_encoder.categories_[1].tolist()}")

Categorical Encoder Fitted
    Input columns   : category, dataset
    Output shape    : (99468, 11)
    Category values : ['Business', 'Entertainment', 'Health', 'News', 'Politics', 'Science', 'Sports', 'Technology']
    Dataset values  : ['bhavikjikadara', 'mahdimashayekhi', 'shawkyelgendy']


# Sample Weight Computation

The objective function $Z$ uses two complementary weight terms:
1. Category Weights, $C_4$
2. Class Weights, $C_5$


## $C_4$: Category Weights

Adjusts weights for over-represented categories (e.g. Sports = 44% of data) so that rarer categories (e.g. Science, Business) contributes proportionally. <br>

$$α_c = \frac{1}{freq(c_i)}$$

In [8]:
cat_freq = df['category'].value_counts(normalize=True).reset_index()
cat_freq.columns = ['category', 'freq']
cat_freq['alpha'] = 1.0 / cat_freq['freq']

category_weight_per_sample = (
    df[['category']].merge(
        cat_freq[['category', 'alpha']],
        on='category',
        how='left')['alpha']
)

print(f"\nCategory weight")
for row in cat_freq.to_pandas().itertuples():
    print(f"    {row.category:<20s}: alpha = {row.alpha:.4f}  (freq = {row.freq:.3f})")


Category weight
    Sports              : alpha = 2.2766  (freq = 0.439)
    Politics            : alpha = 4.5976  (freq = 0.218)
    News                : alpha = 5.0208  (freq = 0.199)
    Health              : alpha = 34.0411  (freq = 0.029)
    Entertainment       : alpha = 34.4299  (freq = 0.029)
    Technology          : alpha = 34.5135  (freq = 0.029)
    Business            : alpha = 34.9133  (freq = 0.029)
    Science             : alpha = 35.6644  (freq = 0.028)


## $C_5$: Class Weights

Corrects for class imbalance between real and fake news. <br>

$$w_1 = \frac{N}{2N_1}, \qquad w_0 = \frac{N}{2N_0}$$

In [9]:
N = len(df)
label_counts = df['label'].value_counts()
N0 = int(label_counts[0])
N1 = int(label_counts[1])

# C5: w_i = N / (2 * N_i)
w0 = N / (2 * N0)
w1 = N / (2 * N1)

class_weight_map = {0: w0, 1: w1}
class_weight_per_sample = df['label'].map(class_weight_map)

print(f"Class weights")
print(f"    w0 (fake) : {w0:.4f}")
print(f"    w1 (real) : {w1:.4f}")

Class weights
    w0 (fake) : 1.0562
    w1 (real) : 0.9495


## Sample Weight

The final per-sample weight is the product $w_{y_i} ⋅ α_{c_i}$, which is passed as `sample_weight` to each base learner and the meta-learner.

In [10]:
sample_weights_cudf = class_weight_per_sample * category_weight_per_sample # Combine sample weights
sample_weights_cudf = sample_weights_cudf / sample_weights_cudf.mean() # Normalise so weights sum to N
sample_weights = to_numpy(sample_weights_cudf) # Convert to numpy for sklean/cuML

df['sample_weight'] = sample_weights_cudf

print(f"\nSample weights stats")
print(f"    min  : {sample_weights.min():.4f}")
print(f"    max  : {sample_weights.max():.4f}")
print(f"    mean : {sample_weights.mean():.4f}")


Sample weights stats
    min  : 0.2700
    max  : 4.7043
    mean : 1.0000


# Stratified Train/Test Split

Train data will comprise of 80% of the data. <br>
Test data will comprise of 20% of the data. Test data is never touched during model training or OOF generation. <br>

Stratification is performed jointly on `label` and `category` to ensure both class and category distributions are preserved in both splits. This prevents distribution shift between training and evaluation. <br>

After splitting, the TF-IDF vectorizer is fitted only on training data and used to transform both splits preventing any leakage of test vocabulary into feature weights.

In [11]:
# Create a joint stratification key: label + category
strat_key = (df['label'].astype(str) + '_' + df['category']).to_pandas()

idx = np.arange(len(df))
idx_train, idx_test = train_test_split(
    idx,
    test_size=0.20,
    random_state=42,
    stratify=strat_key
)

df_train = df.iloc[idx_train].reset_index(drop=True)
df_test = df.iloc[idx_test].reset_index(drop=True)
sw_train = sample_weights[idx_train]
sw_test = sample_weights[idx_test]

# Fit TF-IDF on train only, transform both splits
X_train_tfidf = tfidf.fit_transform(df_train['content'].to_pandas())
X_test_tfidf = tfidf.transform(df_test['content'].to_pandas())

# Categorical features - encoder already fitted on full data
X_train_cat = cat_encoder.transform(df_train[['category', 'dataset']].to_pandas())
X_test_cat = cat_encoder.transform(df_test[['category', 'dataset']].to_pandas())

# Full feature matrices
X_train = sp.hstack([X_train_tfidf, X_train_cat], format='csr')
X_test = sp.hstack([X_test_tfidf, X_test_cat], format='csr')

Y_train = to_numpy(df_train['label'])
Y_test = to_numpy(df_test['label'])

print(f"Train size : {len(df_train):>7,}")
print(f"Test size  : {len(df_test):>6,}")
print(f"X_train    : {X_train.shape}")
print(f"X_test     : {X_test.shape}")
print(f"\nTrain label distribution")
print(f"    Fake: {(Y_train==0).sum():,}")
print(f"    Real: {(Y_train==1).sum():,}")
print(f"\nTest label distribution")
print(f"    Fake: {(Y_test==0).sum():,}")
print(f"    Real: {(Y_test==1).sum():,}")

Train size :  79,574
Test size  : 19,894
X_train    : (79574, 100011)
X_test     : (19894, 100011)

Train label distribution
    Fake: 37,671
    Real: 41,903

Test label distribution
    Fake: 9,418
    Real: 10,476


## Dense Input Setup

A `TruncatedSVD` (Latent Semantic Analysis) projection is applied to reduce the 100,000 dimensional sparse TF-IDF matrix to `SVD_COMPONENTS = 300` dense dimensions before passing it to the cuML models. This keeps GPU memory usage tractable while retaining the dominant semantic structure of the feature space.

- The SVD is **fitted on training data only** and applied to both splits, preventing test-set leakage into the projection.
- Output dtype is cast to float32 for cuML compatibility.
- The cumulative explained variance ratio is printed as a sanity check.


In [12]:
SVD_COMPONENTS = 300

svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=42)
X_train_dense = svd.fit_transform(X_train).astype(np.float32)
X_test_dense = svd.transform(X_test).astype(np.float32)

explained_var = svd.explained_variance_ratio_.sum()

print(f'TruncatedSVD : {SVD_COMPONENTS} components')
print(f'Explained variance ratio : {explained_var:.4f}')
print(f'X_train_dense shape : {X_train_dense.shape}')
print(f'X_test_dense shape : {X_test_dense.shape}')

TruncatedSVD : 300 components
Explained variance ratio : 0.6579
X_train_dense shape : (79574, 300)
X_test_dense shape : (19894, 300)


# K-Fold OOF Framework

The stacking meta-learner $g$ is trained on Out-of-Fold (OOF) predictions from the base models. <br>
How it works:
1. The training set is split into $K$ Folds
2. Each base model is trained on $K - 1$ Folds and predicts on the $K^{th}$ fold
3. After $K$ rotations, every training sample has exactly one OOF prediction made by a model that never saw that sample during training

This ensures the meta-learner is trained on unbiased base-model outputs. We use Stratified K-Fold ($K = 5$) so each fold preserves class and category balance. <br>

A helper function `generate_oof` runs the stratified K-fold OOF loop for a fitted-style estimator <br>

Parameters <br>
- estimator : sklearn estimator with predict_proba
- X : sparse feature matrix (train split)
- Y : label array
- sample_weight : per-sample weights
- cv : StratifiedKFold splitter

Returns <br>
- oof_proba : ndarray of shape (n_samples,) - $P(real \mid x)$ for each OOF sample
- fold_models : list of fitted clones

In [13]:
warnings.filterwarnings('ignore')

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)


def clone_est(estimator):
    try:
        return clone(estimator)
    except Exception:
        return copy.deepcopy(estimator)


def generate_oof(estimator, X, Y, sw, cv):
    oof_proba = np.zeros(len(Y))
    fold_models = []

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, Y), 1):
        model = clone_est(estimator)
        model.fit(X[tr_idx], Y[tr_idx], sample_weight=sw[tr_idx])
        oof_proba[val_idx] = to_numpy(model.predict_proba(X[val_idx]))[:, 1]
        fold_models.append(model)
        print(f'    Fold {fold}/{N_SPLITS} done')

    print()
    return oof_proba, fold_models


print(f"OOF Framework ready : {N_SPLITS}-fold StratifiedKFold")
print(f"Helper Registered : generate_oof()")

OOF Framework ready : 5-fold StratifiedKFold
Helper Registered : generate_oof()


A custom helper `generate_oof_nosw` (no sample-weight) is used for cuML models that do not expose a `sample_weight` argument in `fit`.

In [14]:
def generate_oof_nosw(estimator, X, Y, cv):
    oof_proba = np.zeros(len(Y))
    fold_models = []
    
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, Y), 1):
        model = clone_est(estimator)
        model.fit(X[tr_idx], Y[tr_idx])
        oof_proba[val_idx] = to_numpy(model.predict_proba(X[val_idx]))[:, 1]
        fold_models.append(model)
        print(f'    Fold {fold}/{N_SPLITS} done')
    
    print()
    return oof_proba, fold_models

print(f"Helper Registered : generate_oof_nosw()")

Helper Registered : generate_oof_nosw()


# Weak Learner | Logistic Regression

Logistic Regression is the first base learner. Its log-loss objective aligns directly with the optimisation problem's cross-entropy term. The regularisation parameter $C = \frac{1}{\lambda}$ controls the $\Omega(\theta)$ term.

We use `cuml.linear_model.LogisticRegression` which runs the Quasi-Newton `qn` solver and accepts SciPy CSR matrices directly.

We run a Manual CV Grid Search to orchestrate cuML estimators directly.

In [15]:
# Logistic Regression - Hyperparameter Tuning
lr_C_values = [5.0, 10.0, 15.0, 20.0]
lr_penalty_values = ['l2']

best_lr_auc = -1
best_lr_params = {}

inner_cv_lr = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

for C_val in lr_C_values:
    for penalty_val in lr_penalty_values:
        cv_aucs = []
        for tr_idx, val_idx in inner_cv_lr.split(X_train, Y_train):
            lr_t = cuLR(
                C=C_val,
                penalty=penalty_val,
                solver='qn',
                max_iter=1000
            )
            lr_t.fit(
                X_train[tr_idx],
                Y_train[tr_idx],
                sample_weight=sw_train[tr_idx]
            )
            proba = to_numpy(lr_t.predict_proba(X_train[val_idx]))[:, 1]
            cv_aucs.append(roc_auc_score(Y_train[val_idx], proba))

        mean_auc = np.mean(cv_aucs)
        print(f'C = {C_val:<4.1f} | penalty = {penalty_val:<2s} | AUC = {mean_auc:.4f}')
        if mean_auc > best_lr_auc:
            best_lr_auc = mean_auc
            best_lr_params = {
                'C': C_val,
                'penalty': penalty_val
            }

print(f'\nBest LR params : {best_lr_params}')
print(f'Best CV AUC : {best_lr_auc:.4f}')

lr_best = cuLR(
    C=best_lr_params['C'],
    penalty=best_lr_params['penalty'],
    solver='qn',
    max_iter=1000
)

C = 5.0  | penalty = l2 | AUC = 0.9611
C = 10.0 | penalty = l2 | AUC = 0.9612
C = 15.0 | penalty = l2 | AUC = 0.9608
C = 20.0 | penalty = l2 | AUC = 0.9606

Best LR params : {'C': 10.0, 'penalty': 'l2'}
Best CV AUC : 0.9612


Run the OOF Framework with the best hyperparameters to produce unbiased probability estimates.

In [16]:
print('Generating Logistic Regression OOF Probabilities')
lr_oof, lr_fold_models = generate_oof(lr_best, X_train, Y_train, sw_train, skf)

lr_oof_auc = roc_auc_score(Y_train, lr_oof)
print(f'Logistic Regression OOF AUC : {lr_oof_auc:.4f}')

Generating Logistic Regression OOF Probabilities
    Fold 1/5 done
    Fold 2/5 done
    Fold 3/5 done
    Fold 4/5 done
    Fold 5/5 done

Logistic Regression OOF AUC : 0.9624


# Weak Learner | Complement Naive Bayes

Complement Naive Bayes estimates the complement of each class rather than the class itself, which corrects for skewed class and category distributions - Exactly the imbalance thiis dataset exhibits (Sports at ~44% of records).

It accepts SciPy CSR sparse matrices, supports `sample_weight` in `fit` and outputs calibrated probabilities via `predict_proba`.

ComplementNB requires non-negative inputs; sublinear TF-IDF satisfies this.

In [17]:
cnb_alphas = [1e-05, 1e-04, 1e-03, 1e-02, 0.1, 1.0]
best_cnb_auc = -1
best_cnb_alpha = None

inner_cv_cnb = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)

for alpha_val in cnb_alphas:
    cv_aucs = []
    for tr_idx, val_idx in inner_cv_cnb.split(X_train, Y_train):
        X_tr, X_val = X_train_tfidf[tr_idx], X_train_tfidf[val_idx]
        Y_tr, Y_val = Y_train[tr_idx], Y_train[val_idx]
        sw_tr_inner = sw_train[tr_idx]

        cnb = cuCNB(alpha=alpha_val)
        cnb.fit(X_tr, Y_tr, sample_weight=sw_tr_inner)
        proba = to_numpy(cnb.predict_proba(X_val))[:, 1]
        proba = np.nan_to_num(proba, nan=0.5, posinf=1.0, neginf=0.0) # replace Nan/Inf with neutral probability
        cv_aucs.append(roc_auc_score(Y_val, proba))

    mean_auc = np.mean(cv_aucs)
    print(f'alpha = {alpha_val:.2e} | AUC = {mean_auc:.4f}')
    if mean_auc > best_cnb_auc:
        best_cnb_auc = mean_auc
        best_cnb_alpha = alpha_val

print(f'\nBest ComplementNB alpha : {best_cnb_alpha}')
print(f'Best CV AUC : {best_cnb_auc:.4f}')

cnb_best = cuCNB(alpha=best_cnb_alpha)

alpha = 1.00e-05 | AUC = 0.9504
alpha = 1.00e-04 | AUC = 0.9499
alpha = 1.00e-03 | AUC = 0.9491
alpha = 1.00e-02 | AUC = 0.9479
alpha = 1.00e-01 | AUC = 0.9460
alpha = 1.00e+00 | AUC = 0.9431

Best ComplementNB alpha : 1e-05
Best CV AUC : 0.9504


Run the OOF Framework with the best hyperparameters to produce unbiased probability estimates.

In [18]:
print('Generating ComplementNB OOF Probabilities')
cnb_oof, cnb_fold_models = generate_oof(cnb_best, X_train_tfidf, Y_train, sw_train, skf)

cnb_oof_auc = roc_auc_score(Y_train, cnb_oof)
print(f'ComplementNB OOF AUC : {cnb_oof_auc:.4f}')

Generating ComplementNB OOF Probabilities
    Fold 1/5 done
    Fold 2/5 done
    Fold 3/5 done
    Fold 4/5 done
    Fold 5/5 done

ComplementNB OOF AUC : 0.9508


# Weak Learner | Stochastic Gradient Descent (SGD) Classifier

Stochastic Gradient Descent (SGD) `sklearn.linear_model.SGDClassifier` configured with `loss='hinge'` and `penalty='l2'`.

With hinge loss, SGD Classifier implements margin-based updates. This rule only penalises misclassifications and pushes the decision boundary aggressively.

SGD Classifier does not natively output probabilities when `loss='hinge'` , so it is wrapped with `CalibratedClassifierCV` (sigmoid calibration) to produce `predict_proba` output compatible with the stacking pipeline. Using `loss='hinge'` preserves the margin-based diversity that distinguishes it from the Logistic Regression base learner.

In [19]:
sgd_eta0_values = [0.001, 0.01, 0.1]
sgd_epoch_values = [5, 15]

best_sgd_auc = -1
best_sgd_params = {}

inner_cv_sgd = StratifiedKFold(n_splits=3, shuffle=True, random_state=2)

for eta0_val in sgd_eta0_values:
    for epoch_val in sgd_epoch_values:
        cv_aucs = []
        for tr_idx, val_idx in inner_cv_sgd.split(X_train, Y_train):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            Y_tr, Y_val = Y_train[tr_idx], Y_train[val_idx]
            sw_tr_inner = sw_train[tr_idx]

            sgd = skSGD(
                loss='hinge',
                penalty='l2',
                eta0=eta0_val,
                max_iter=epoch_val,
                learning_rate='constant',
                random_state=42
            )
            cal = CalibratedClassifierCV(sgd, method='sigmoid', cv=3)
            cal.fit(X_tr, Y_tr, sample_weight=sw_tr_inner)
            proba = cal.predict_proba(X_val)[:, 1]
            cv_aucs.append(roc_auc_score(Y_val, proba))

        mean_auc = np.mean(cv_aucs)
        print(f'eta0 = {eta0_val:.4f} | epochs = {epoch_val:>2d} | AUC = {mean_auc:.4f}')
        if mean_auc > best_sgd_auc:
            best_sgd_auc = mean_auc
            best_sgd_params = {
                'eta0': eta0_val,
                'max_iter': epoch_val
            }

print(f'\nBest SGD Classifier params : {best_sgd_params}')
print(f'Best CV AUC : {best_sgd_auc:.4f}')

sgd_best = CalibratedClassifierCV(
    skSGD(
        loss='hinge',
        penalty='l2',
        learning_rate='constant',
        random_state=42,
        **best_sgd_params
    ),
    method = 'sigmoid',
    cv = 3
)

eta0 = 0.0010 | epochs =  5 | AUC = 0.7190
eta0 = 0.0010 | epochs = 15 | AUC = 0.8249
eta0 = 0.0100 | epochs =  5 | AUC = 0.9348
eta0 = 0.0100 | epochs = 15 | AUC = 0.9442
eta0 = 0.1000 | epochs =  5 | AUC = 0.9169
eta0 = 0.1000 | epochs = 15 | AUC = 0.9250

Best SGD Classifier params : {'eta0': 0.01, 'max_iter': 15}
Best CV AUC : 0.9442


Run the OOF Framework with the best hyperparameters to produce unbiased probability estimates.

In [20]:
print('Generating SGD Classifier OOF Probabilities')
sgd_oof, sgd_fold_models = generate_oof(sgd_best, X_train, Y_train, sw_train, skf)

sgd_oof_auc = roc_auc_score(Y_train, sgd_oof)
print(f'SGD Classifier OOF AUC : {sgd_oof_auc:.4f}')

Generating SGD Classifier OOF Probabilities
    Fold 1/5 done
    Fold 2/5 done
    Fold 3/5 done
    Fold 4/5 done
    Fold 5/5 done

SGD Classifier OOF AUC : 0.9447


# Weak Learner | K-Nearest Neighbors (KNN)

K-Nearest Neighbors classifies each sample by majority vote of its $k$ nearest neighbors in the SVD-projected feature space. It is a **non-parametric, instance-based** learner: unlike the linear models (LR, MBSDG) and the probabilistic CNB, KNN makes no distributional assumption and captures local neighborhood structure in the latent semantic space. This inductive bias provides **complementary diversity** to the existing base learners.

Two key parameters are tuned via manual inner 3-fold CV:
- `n_neighbors`: controls the size of the voting neighborhood
- `metric`: `euclidian` vs `cosine`; cosine distance is generally better suited to TF-IDF derived embeddings because it is magnitude-invariant.

`cuml.neighbors.KNeighborsClassifier` requires dense input and does not accept `sample_weight` in `fit`; sample weighting is therefore not applied in this learner.

In [21]:
knn_n_neighbors_values = [25, 50, 75, 100]
knn_metric_values = ['euclidean', 'cosine']

best_knn_auc = -1
best_knn_params = {}

inner_cv_knn = StratifiedKFold(n_splits=3, shuffle=True, random_state=3)

for n_val in knn_n_neighbors_values:
    for m_val in knn_metric_values:
        cv_aucs = []
        for tr_idx, val_idx in inner_cv_knn.split(X_train_dense, Y_train):
            knn_t = cuKNN(
                n_neighbors=n_val,
                metric=m_val
            )
            knn_t.fit(X_train_dense[tr_idx], Y_train[tr_idx])
            proba = to_numpy(knn_t.predict_proba(X_train_dense[val_idx]))[:, 1]
            cv_aucs.append(roc_auc_score(Y_train[val_idx], proba))

        mean_auc = np.mean(cv_aucs)
        print(f'n_neighbors = {n_val:>3d} | metric = {m_val:<9s} | AUC = {mean_auc:,.4f}')
        if mean_auc > best_knn_auc:
            best_knn_auc = mean_auc
            best_knn_params = {
                'n_neighbors': n_val, 
                'metric': m_val
            }

print(f'\nBest KNN params : {best_knn_params}')
print(f'Best CV AUC : {best_knn_auc:.4f}')

knn_best = cuKNN(**best_knn_params)

n_neighbors =  25 | metric = euclidean | AUC = 0.9476
n_neighbors =  25 | metric = cosine    | AUC = 0.9483
n_neighbors =  50 | metric = euclidean | AUC = 0.9480
n_neighbors =  50 | metric = cosine    | AUC = 0.9488
n_neighbors =  75 | metric = euclidean | AUC = 0.9476
n_neighbors =  75 | metric = cosine    | AUC = 0.9485
n_neighbors = 100 | metric = euclidean | AUC = 0.9471
n_neighbors = 100 | metric = cosine    | AUC = 0.9479

Best KNN params : {'n_neighbors': 50, 'metric': 'cosine'}
Best CV AUC : 0.9488


Run the OOF Framework with the best hyperparameters to produce unbiased probability estimates. `generate_oof_nosw` is used because `KNeighborsClassifier` does not support `sample_weight`.

In [22]:
print('Generating KNN OOF Probabilities')
knn_oof, knn_fold_models = generate_oof_nosw(knn_best, X_train_dense, Y_train, skf)

knn_oof_auc = roc_auc_score(Y_train, knn_oof)
print(f'KNN OOF AUC : {knn_oof_auc:.4f}')

Generating KNN OOF Probabilities
    Fold 1/5 done
    Fold 2/5 done
    Fold 3/5 done
    Fold 4/5 done
    Fold 5/5 done

KNN OOF AUC : 0.9490


# Weak Learner | Random Forest

Random Forest is a bagging ensemble of decision trees. Each tree is trained on a bootstrap sample of the data and considers only a random subset of features at each split. Predictions are averaged (soft-voting) across all trees.

Its **non-linear, tree-based** inductive bias contrasts sharply with the linear learners (LR, SGD) and the probabilistic learners (CNB, KNN), maximising the diversity of the base learner pool and the potential gain from the meta-learner.

Three hyperparameters are tuned via manual inner 3-fold CV:
- `n_estimators`: number of trees in the forest
- `max_depth`: maximum depth of each tree (controls bias-variance trade-off)
- `max_features`: fraction of features considered at each split (`sqrt` or `log`)

`cuml.ensemble.RandomForestClassifier` requires dense input and does not accept `sample_weight` in `fit`; sample weighting is therefore not applied for this learner.

In [23]:
rf_n_estimators_values = [100, 150, 200]
rf_max_depth_values = [10, 15, 20]
rf_max_features_values = ['sqrt']

best_rf_auc = -1
best_rf_params = {}

inner_cv_rf = StratifiedKFold(n_splits=3, shuffle=True, random_state=4)

for n_est in rf_n_estimators_values:
    for max_d in rf_max_depth_values:
        for max_f in rf_max_features_values:
            cv_aucs = []
            for tr_idx, val_idx in inner_cv_rf.split(X_train_dense, Y_train):
                rf_t = cuRF(
                    n_estimators=n_est,
                    max_depth=max_d,
                    max_features=max_f,
                    random_state=42
                )
                rf_t.fit(X_train_dense[tr_idx], Y_train[tr_idx])
                proba = to_numpy(rf_t.predict_proba(X_train_dense[val_idx]))[:, 1]
                cv_aucs.append(roc_auc_score(Y_train[val_idx], proba))
            
            mean_auc = np.mean(cv_aucs)
            print(f'n_estimators = {n_est:>3d} | max_depth = {max_d:>2d} | max_features = {max_f:<4s} | AUC = {mean_auc:.4f}')
            if mean_auc > best_rf_auc:
                best_rf_auc = mean_auc
                best_rf_params = {
                    'n_estimators': n_est,
                    'max_depth': max_d,
                    'max_features': max_f
                }

print(f'\nBest RF params : {best_rf_params}')
print(f'Best CV AUC : {best_rf_auc:.4f}')

rf_best = cuRF(
    random_state=42,
    **best_rf_params
)

n_estimators = 100 | max_depth = 10 | max_features = sqrt | AUC = 0.9488
n_estimators = 100 | max_depth = 15 | max_features = sqrt | AUC = 0.9552
n_estimators = 100 | max_depth = 20 | max_features = sqrt | AUC = 0.9565
n_estimators = 150 | max_depth = 10 | max_features = sqrt | AUC = 0.9492
n_estimators = 150 | max_depth = 15 | max_features = sqrt | AUC = 0.9556
n_estimators = 150 | max_depth = 20 | max_features = sqrt | AUC = 0.9572
n_estimators = 200 | max_depth = 10 | max_features = sqrt | AUC = 0.9493
n_estimators = 200 | max_depth = 15 | max_features = sqrt | AUC = 0.9557
n_estimators = 200 | max_depth = 20 | max_features = sqrt | AUC = 0.9572

Best RF params : {'n_estimators': 200, 'max_depth': 20, 'max_features': 'sqrt'}
Best CV AUC : 0.9572


Run the OOF Framework with the best hyperparameters to produce unbiased probability estimates. `generate_oof_nosw` is used because `RandomForestClassifier` does not support `sample_weight`.

In [24]:
print('Generating Random Forest OOF Probabilities')
rf_oof, rf_fold_models = generate_oof_nosw(rf_best, X_train_dense, Y_train, skf)

rf_oof_auc = roc_auc_score(Y_train, rf_oof)
print(f'Random Forest OOF AUC : {rf_oof_auc:.4f}')

Generating Random Forest OOF Probabilities
    Fold 1/5 done
    Fold 2/5 done
    Fold 3/5 done
    Fold 4/5 done
    Fold 5/5 done

Random Forest OOF AUC : 0.9577


# Meta-Feature Matrix Assembly

The meta-feature matrix $\mathbf{M}$ is assembled by horizontally stacking the OOF probability columns from each base learner:

$$
\mathbf{M} = [\hat{p}_{LR},\ \hat{p}_{CNB},\ \hat{p}_{SGD},\ \hat{p}_{KNN},\ \hat{p}_{RF}]
\in \mathbb{R}^{N_{train} \times 5}
$$

Each column is an unbiased estimate of $P(y=1 \mid x_i)$ from a different model family. The meta-learner $g$ learns to optimally combine these five signals. We also compute pairwise correlations to check whether the base learners are sufficiently diverse (low correlation → higher ensemble benefit)

In [25]:
oof_list = [
    lr_oof,
    cnb_oof,
    sgd_oof,
    knn_oof,
    rf_oof
]

M_train = np.column_stack(oof_list)

base_learner_names = [
    'LR',
    'CNB',
    'SGD',
    'KNN',
    'RF'
]

print(f'Meta-feature matrix shape: {M_train.shape}')

print(f'OOF AUC per base learner:')
for name, oof in zip(base_learner_names, oof_list):
    print(f'    {name:<22s} : {roc_auc_score(Y_train, oof):.4f}')

print("\nPairwise correlation of OOF probabilities:")
corr_df = pd.DataFrame(M_train, columns=base_learner_names).corr()
print(corr_df.round(3))

Meta-feature matrix shape: (79574, 5)
OOF AUC per base learner:
    LR                     : 0.9624
    CNB                    : 0.9508
    SGD                    : 0.9447
    KNN                    : 0.9490
    RF                     : 0.9577

Pairwise correlation of OOF probabilities:
        LR    CNB    SGD    KNN     RF
LR   1.000  0.887  0.945  0.906  0.927
CNB  0.887  1.000  0.887  0.855  0.892
SGD  0.945  0.887  1.000  0.908  0.937
KNN  0.906  0.855  0.908  1.000  0.950
RF   0.927  0.892  0.937  0.950  1.000


# Meta-Learner Training

The meta-learner $g$ maps stacked OOF probabilities to a final class prediction. We use **Logistic Regression** as the meta-learner because:

- It is interpretable i.e., the coefficients show each base model's contribution
- It prevents overfitting on the small (5-dimensional) meta-feature matrix
- Its log-loss objective directly optimises the same criterion as the stacking objective function

The meta learner is `cuml.linear_model.LogisticRegression` with the `qn` solver. We pass the sample weights to the meta-learner as well, maintaining the class and category correction all the way through the pipeline. The meta-feature matrix $\mathbf{M}$ is small ($N_{train} \times 5$ dense), so it can be passed as a NumPy array directly to cuML. The meta-learner is tuned over $C$ using a manual 5-fold cross-validation on the meta-feature matrix.

In [34]:
meta_C_values = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]

best_meta_auc = -1
best_meta_C = None

meta_cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for C_val in meta_C_values:
    cv_aucs = []
    for tr_idx, val_idx in meta_cv_splitter.split(M_train, Y_train):
        meta_t = cuLR(
            C=C_val,
            penalty='l2',
            solver='qn',
            max_iter=1000
        )
        meta_t.fit(
            M_train[tr_idx],
            Y_train[tr_idx],
            sample_weight=sw_train[tr_idx]
        )
        proba = to_numpy(meta_t.predict_proba(M_train[val_idx]))[:, 1]
        cv_aucs.append(roc_auc_score(Y_train[val_idx], proba))
    
    mean_auc = np.mean(cv_aucs)
    print(f'    C = {C_val:<5.2f} | AUC = {mean_auc:.4f}')
    if mean_auc > best_meta_auc:
        best_meta_auc = mean_auc
        best_meta_C = C_val

print(f'\nBest meta-learner C : {best_meta_C}')
print(f'Best meta-learner AUC : {best_meta_auc:.4f}')

    C = 0.01  | AUC = 0.9640
    C = 0.10  | AUC = 0.9635
    C = 0.50  | AUC = 0.9631
    C = 1.00  | AUC = 0.9630
    C = 5.00  | AUC = 0.9630
    C = 10.00 | AUC = 0.9630

Best meta-learner C : 0.01
Best meta-learner AUC : 0.9640


In [35]:
meta_learner = cuLR(
    C=best_meta_C,
    penalty='l2',
    solver='qn',
    max_iter=1000
)

meta_learner.fit(
    M_train,
    Y_train,
    sample_weight=sw_train
)

print(f'Meta-learner coefficients (base model weights):')
coefs = to_numpy(meta_learner.coef_).flatten()
for name, coef in zip(base_learner_names, coefs):
    print(f'    {name:<3s} : {coef:+.4f}')

Meta-learner coefficients (base model weights):
    LR  : +0.8816
    CNB : +1.5187
    SGD : -0.0701
    KNN : +2.2182
    RF  : +2.9664


# Full Retraining of Base Models on Train Set

The OOF fold models were trained on $\frac{K-1}{K}$ of the training data. For final test-set inference we retrain each base learner on the **full training set** to maximise the signal available before generating test meta-features.

Each base model is retrained with its best hyperparameters found. 

In [37]:
print("Retraining base models on full training set ...")

# Weak Learner | Logistic Regression
lr_final = cuLR(
    C=best_lr_params['C'],
    penalty=best_lr_params['penalty'],
    solver='qn',
    max_iter=1000
)
lr_final.fit(X_train, Y_train, sample_weight=sw_train)
print('[1/5] Logistic Regression - done')

# Weak Learner | Complement Naive Bayes
cnb_final = cuCNB(alpha=best_cnb_alpha)
cnb_final.fit(X_train_tfidf, Y_train, sample_weight=sw_train)
print('[2/5] ComplementNB - done')

# Weak Learner | SGD Classifier
sgd_final = CalibratedClassifierCV(
    skSGD(
        loss='hinge',
        penalty='l2',
        random_state=42,
        **best_sgd_params
    ),
    method='sigmoid',
    cv=3
)
sgd_final.fit(X_train, Y_train, sample_weight=sw_train)
print('[3/5] SGD Classifier - done')

# Weak Learner | KNN
knn_final = cuKNN(**best_knn_params)
knn_final.fit(X_train_dense, Y_train)
print('[4/5] KNN - done')

# Weak Learner | Random Forest
rf_final = cuRF(
    random_state=42,
    **best_rf_params
)
rf_final.fit(X_train_dense, Y_train)
print('[5/5] Random Forest - done')

print('\nAll base models retrained on full training set')

Retraining base models on full training set ...
[1/5] Logistic Regression - done
[2/5] ComplementNB - done
[3/5] SGD Classifier - done
[4/5] KNN - done
[5/5] Random Forest - done

All base models retrained on full training set


# Test Set Inference & Evaluation

The full prediction pipeline for the test set is:

1. Each retrained base model generates $P(y=1 \mid x_i)$ for test samples → assembles the **test meta-feature matrix** $\mathbf{M}_{test}$
2. The meta-learner $g$ maps $\mathbf{M}_{test}$ to final probabilities
3. Threshold at 0.5 for hard predictions

We report AUC-ROC, Accuracy, Precision, Recall, and F1-score on the held-out test set. We also compare against each individual base learner to demonstrate the stacking improvement.

Generate test set meta-features from retrained base models

In [39]:
lr_test_proba = to_numpy(lr_final.predict_proba(X_test))[:, 1]
cnb_test_proba = to_numpy(cnb_final.predict_proba(X_test_tfidf))[:, 1]
sgd_test_proba = to_numpy(sgd_final.predict_proba(X_test))[:, 1]
knn_test_proba = to_numpy(knn_final.predict_proba(X_test_dense))[:, 1]
rf_test_proba = to_numpy(rf_final.predict_proba(X_test_dense))[:, 1]

M_test = np.column_stack(
    [
        lr_test_proba, 
        cnb_test_proba,
        sgd_test_proba,
        knn_test_proba,
        rf_test_proba
    ]
)

Meta-learner final probabilities

In [40]:
stack_test_proba = to_numpy(meta_learner.predict_proba(M_test))[:, 1]
stack_test_pred = (stack_test_proba >= 0.5).astype(int)

Evaluate Model

In [ ]:
def evaluate_model(name, y_true, y_proba):
    y_pred = (y_proba >= 0.5).astype(int)
    return {
        'Model'     : name,
        'AUC'       : roc_auc_score(y_true, y_proba),
        'Accuracy'  : accuracy_score(y_true, y_pred),
        'Precision' : precision_score(y_true, y_pred),
        'Recall'    : recall_score(y_true, y_pred),
        'F1'        : f1_score(y_true, y_pred)
    }



___